# PHASE 2 — MLP từ Lab4 và CNN trên MNIST

> **Đồ án 1 (giữa kỳ) — Nhận dạng chữ số viết tay MNIST**  
> **Trọng tâm:** MLP tự cài backpropagation theo luồng `Lab4.ipynb`; bổ sung CNN đơn giản để so sánh.  
> **Hình thức:** Đây là **sườn phân việc**, chỉ gồm các ô Markdown `TODO`, **chưa cài đặt code**.

**Tài liệu:** `DoAn_ThucHanh_NhanDang.pdf` (Đồ án 1, tr. 1–2; Phụ lục tr. 4); `Lab4.ipynb` (mẫu MLP chính); `phase1.ipynb` (giao thức dữ liệu); `NaiveBayes(1).ipynb`, `BTVN_Lab01.ipynb`, `TL_Lab03.ipynb` (format/metrics/so sánh).

**Luồng thực hiện:** Dữ liệu → forward → backward → kiểm gradient → SGD + momentum → huấn luyện MLP → thí nghiệm & chọn mô hình → đánh giá 3 tập → CNN → so sánh → xuất kết quả.


## 0. Mục tiêu và phạm vi

**TODO:**
1. **MLP bắt buộc:** tự triển khai lan truyền ngược; trình bày rõ `forward → loss → backward → update` theo `Lab4.ipynb`.
2. **CNN bổ sung theo yêu cầu đồ án này:** tự thiết kế CNN nhỏ, huấn luyện, đánh giá và so sánh với MLP. Trong PDF, CNN thuộc phần **nâng cao**, không thay một trong 4 phương pháp bắt buộc.
3. Mỗi mô hình cần có **accuracy, precision/recall/F1 (macro), confusion matrix** trên **train / validation / test**, kèm thời gian huấn luyện và dự đoán.
4. Có **ít nhất một thí nghiệm siêu tham số kèm biểu đồ** trong Phase 2; cuối phase xuất kết quả cho Phase 3 tổng hợp.

**Quy tắc from scratch của Lab4:** được dùng PyTorch để lưu trọng số/dựng các lớp cho MLP; **tự viết gradient, backward và SGD + momentum**. `autograd` dùng để **kiểm chứng riêng**, không thay phần backprop chính. Đối chiếu mô hình dựng sẵn chỉ là baseline kiểm chứng.

**Điều chỉnh so với Lab4:** giữ **cùng split của Phase 1 (54.000/6.000/10.000, `SEED=42`, stratified)**, không dùng cách chia 50.000/10.000 ngẫu nhiên của lab; đây là quy ước đồ án để các phase so sánh công bằng.


### 0.1. Chuẩn bị môi trường và nguyên tắc tái lập

**TODO:**
1. Import NumPy, Matplotlib, pandas, `torch`/`torch.nn` (phần khung), công cụ đo thời gian và tính metrics.
2. Cố định `SEED=42`; ghi thiết bị CPU/GPU, phiên bản thư viện và cách chạy notebook từ đầu.
3. Tạo/thống nhất thư mục `artifacts/`, `models/`, `results/`, `figures/`; đặt mã mô hình `MLP`, `CNN` đúng `plan.md`.
4. Với MLP: không sử dụng `loss.backward()` hoặc `torch.optim` làm bộ huấn luyện chính; chỉ bật autograd trong ô đối chiếu gradient như ở Lab4.

**Tái sử dụng:** `Lab4.ipynb` phần import + seed; `TL_Lab03.ipynb` phần đo thời gian.


## 1. Dữ liệu MNIST dùng chung

### 1.1. Nạp dữ liệu và giữ nguyên train / validation / test

**TODO:**
1. Tái sử dụng `load_mnist` từ `Lab4.ipynb` để tải **MNIST 28×28**: 60.000 train gốc và 10.000 test gốc.
2. **Bắt buộc đọc** `artifacts/split_indices.npz` do `phase1.ipynb` tạo, lấy `train_idx`, `val_idx`, `test_idx` theo **đúng thứ tự lưu**: 54.000 train, 6.000 validation, 10.000 test. Nếu thiếu/không hợp lệ, báo lỗi và yêu cầu chạy bước chia tập ở Phase 1; **không chia lại chỉ dựa vào SEED**.
3. Trước khi áp chỉ số, kiểm `train_source_sha256`/`test_source_sha256` của **ảnh + nhãn gốc theo đúng thứ tự** khớp file; kiểm shape, không trùng/mất chỉ số, nhãn `0..9` và phân bố 3 tập. Chỉ train được dùng để fit các tham số tiền xử lý.
4. Không sử dụng tập test để chọn mô hình/siêu tham số; chỉ đánh giá test **sau khi khóa cấu hình cuối** của từng mô hình.

**Khác Lab4:** điều chỉnh đoạn tạo `tr_idx/va_idx`, không bê nguyên tỷ lệ 50k/10k.


### 1.2. Tiền xử lý và kiểm tra ảnh đầu vào

**TODO:**
1. Sau khi lấy mẫu theo **đúng thứ tự `train_idx`/`val_idx`/`test_idx`**, chuyển pixel sang `float32`, chia `255.0`; giữ ảnh dạng `(N,28,28)` và tạo vector `(N,784)` cho MLP.
2. Nếu theo bước z-score của `Lab4`, tính **mean/std chỉ trên train** rồi biến đổi val/test; ghi rõ pipeline tiền xử lý riêng của **MLP** và **CNN** (chỉ /255 hay thêm chuẩn hóa từ train), không `fit` lại trên validation/test.
3. Tạo ảnh/tensor CNN `(N,1,28,28)` từ **cùng các chỉ số mẫu và thứ tự nhãn** của MLP; kiểm tra reshape không làm sai chữ số.
4. Minh họa ngắn ảnh 0–9 và biểu đồ phân bố nhãn 3 tập; **không lặp toàn bộ EDA của Phase 1** (các EDA sâu trong Lab4 Phần 8 được chuyển thành phần tham khảo).

**Tái sử dụng:** `preprocess`, `show_images` ở `Lab4.ipynb`; phần dữ liệu/EDA ở `phase1.ipynb`.


## 2. MLP — triển khai theo trình tự Lab4

### 2.1. Kiến trúc mạng và lan truyền thuận

**TODO:**
1. Dùng `build_mlp` của `Lab4` làm **khung lớp** (`nn.Linear`, `nn.Sequential`, activation), không dùng MLP dựng sẵn để thay phần tính toán lõi.
2. Chọn baseline tham khảo **784 → 256 → 128 → 10**, ReLU, He initialization; ghi input/output shape và số tham số.
3. Mô tả ngắn công thức **linear → activation → logits**, softmax chỉ tính trong hàm mất mát.

**Tái sử dụng:** `Lab4.ipynb` Phần 1.1 (`build_mlp`, `n_params`).


### 2.2. `forward_with_cache` — lưu thông tin cho backward

**TODO:**
1. Hoàn thiện `forward_with_cache(model, X, train=True)` trả `logits` và `cache` cho từng layer: `(layer, a_in, a_out, extra)`.
2. Nếu dùng dropout, tự xử lý **inverted dropout** và lưu mask để backward; tắt dropout khi validation/test.
3. Kiểm tra đầu ra `forward_with_cache(..., train=False)` khớp `model(X)`, đúng shape `(batch,10)`.

**Tái sử dụng:** `Lab4.ipynb` Phần 1.2 và ô kiểm tra sau hàm; chỉ bổ sung các `TODO` còn trống.


### 2.3. Softmax + cross-entropy và gradient đầu ra

**TODO:**
1. Triển khai `softmax_cross_entropy(logits, y)` trả **loss trung bình** và gradient `dJ/dlogits`.
2. Tính ổn định số học (trừ logit lớn nhất trước `exp`), chuẩn hóa gradient theo số mẫu mini-batch.
3. Test shape, tổng xác suất theo hàng, loss/gradient trên ví dụ nhỏ của Lab4.

**Tái sử dụng:** `Lab4.ipynb` Phần 2.1 + ô kiểm tra (không sao chép lời giải từ autograd).


### 2.4. Backward từng lớp: `nn.Linear`, activation, dropout

**TODO:**
1. Hoàn thiện `linear_backward`: gradient theo `W`, `b` và activation đầu vào bằng phép nhân ma trận; kiểm tra khớp `weight.shape`.
2. Viết đạo hàm **ReLU** (bắt buộc với baseline); nếu giữ các lựa chọn của Lab4 thì bổ sung `sigmoid_backward`, `tanh_backward`.
3. Nếu mô hình dùng dropout, viết `dropout_backward` với **đúng mask** đã lưu ở forward.
4. Kiểm trên một mini-batch nhỏ, tránh chia trung bình gradient **hai lần**.

**Tái sử dụng:** `Lab4.ipynb` Phần 2.2–2.3 (`linear_backward`, các hàm `_backward`).


### 2.5. Ghép vòng lan truyền ngược toàn mạng

**TODO:**
1. Hoàn thiện `backward(cache, grad_out, weight_decay=0.0)`: duyệt các layer từ **cuối về đầu**, truyền gradient qua từng module.
2. Gán `.grad` thủ công cho các tham số `nn.Linear`, **không gọi** `loss.backward()` trong huấn luyện MLP chính.
3. Nếu có L2 (`weight_decay`), cộng gradient cho `W`, **không** áp lên bias; kiểm tra gradient finite (không NaN/Inf).

**Tái sử dụng:** `Lab4.ipynb` Phần 2.4.


### 2.6. Kiểm chứng backward với PyTorch autograd

**TODO:**
1. Lấy một mạng nhỏ và cùng mini-batch, so sánh gradient thủ công với gradient do autograd tính **trong ô kiểm chứng riêng**.
2. Giữ cùng trọng số, dtype và dropout mask (nếu có); báo sai số tuyệt đối/tương đối từng layer.
3. Sửa phần tính gradient trước khi huấn luyện nếu sai số lớn; nêu kết quả kiểm chứng trong notebook.

**Tái sử dụng:** `Lab4.ipynb` Phần 3; autograd **không** là thuật toán train MLP chính.


### 2.7. Tự viết `SGDMomentum` và vòng lặp mini-batch

**TODO:**
1. Hoàn thiện optimizer `SGDMomentum`: trạng thái vận tốc cho từng tham số, cập nhật `weight`/`bias` từ `.grad` thủ công.
2. Hoàn thiện `train_one_epoch`: shuffle train → mini-batch → forward/cache → loss → backward → optimizer.step.
3. Tạo hàm `evaluate` / `train_model` để ghi loss/accuracy theo epoch; khi đánh giá không cập nhật gradient, dropout ở chế độ eval.
4. Kiểm tra optimizer trên ví dụ tham số nhỏ; ghi số epoch, batch size, learning rate, momentum và thời gian huấn luyện.

**Tái sử dụng:** `Lab4.ipynb` Phần 4.1–4.2 (`SGDMomentum`, `train_one_epoch`, `evaluate`, `train_model`).


### 2.8. Huấn luyện MLP baseline và vẽ đường hội tụ

**TODO:**
1. Huấn luyện baseline với cấu hình tham khảo của Lab4 (784–256–128–10, ReLU/He, SGD momentum, mini-batch); điều chỉnh số epoch phù hợp tài nguyên.
2. Vẽ **train loss / validation loss** và **train accuracy / validation accuracy** theo epoch.
3. Kiểm tra dấu hiệu hội tụ, overfitting/underfitting; ghi train time, cấu hình và mốc validation tốt nhất.

**Tái sử dụng:** `Lab4.ipynb` Phần 4.3 + `plot_history`. Mốc val accuracy 97% ở Lab4 là **mục tiêu bài lab**, không phải ngưỡng bắt buộc từ PDF đồ án.


### 2.9. Thí nghiệm siêu tham số MLP — có biểu đồ

**TODO (bắt buộc):**
1. **Chọn ít nhất 1 thí nghiệm:** (a) learning rate/momentum như `Lab4` Phần 5.1, **hoặc** (b) số nơ-ron/số lớp ẩn như Phần 5.3.
2. Giữ nguyên train/validation, seed, số epoch/batch size giữa các cấu hình (trừ tham số đang khảo sát); **chỉ dùng validation để lựa chọn**.
3. Lập bảng các cấu hình với validation **accuracy, F1 macro** và thời gian; vẽ **ít nhất 1 biểu đồ** đường/cột hoặc heatmap có nhãn rõ ràng.
4. Nhận xét mối quan hệ giữa siêu tham số, hội tụ, quá khớp và chi phí huấn luyện.

**Không bắt buộc bê nguyên:** tất cả bài sweep batch size, activation, vanishing gradient, constant/Xavier/He của `Lab4` Phần 5; chỉ mở rộng nếu có thời gian và mục tiêu rõ ràng.


### 2.10. Chọn MLP cuối bằng validation và early stopping

**TODO:**
1. Chọn một cấu hình MLP từ kết quả thí nghiệm, ghi lý do dựa trên **validation**, không xem test.
2. Tái sử dụng/tự hoàn thiện `train_with_early_stopping` ở `Lab4` Phần 6: theo dõi validation, lưu và phục hồi **best checkpoint**.
3. Lưu mô hình, siêu tham số, seed, số epoch dừng; ghi thời gian `train_seconds` chỉ tính phần fit.

**Lưu ý:** test chỉ được đo trên checkpoint cuối sau khi chọn cấu hình; không chạy test trong sweep.


### 2.11. Đánh giá MLP trên train / validation / test

**TODO:**
1. Trên **mỗi** tập train, validation, test: xuất `y_pred`, tính **accuracy, precision_macro, recall_macro, f1_macro**.
2. Vẽ **confusion matrix 10×10** cho từng tập; phân tích 2–3 cặp chữ số nhầm nhiều nhất và ví dụ dự đoán sai.
3. Đo `predict_seconds` theo từng tập, ghi `train_seconds` của checkpoint cuối; nêu chênh lệch train/validation/test.
4. Nếu cần kiểm chứng theo rubric: so sánh ngắn với `sklearn.neural_network.MLPClassifier` trên **cùng split** (có thể subset train nếu máy yếu), ghi rõ khác biệt cấu hình; **không thay** MLP tự backprop.

**Tái sử dụng:** `Lab4.ipynb` Phần 7 (dự đoán, ma trận nhầm lẫn), `TL_Lab03.ipynb` phần bảng metric và time.


## 3. CNN đơn giản — mô hình nâng cao của PDF

### 3.1. Chuẩn bị đầu vào và thiết kế CNN

**TODO:**
1. Dùng ảnh cùng split/nhãn với MLP, nhưng giữ tensor **`(N,1,28,28)`**, không flatten trước lớp convolution.
2. Thiết kế mạng nhỏ **Conv → ReLU → Pool → Conv → ReLU → Pool → Flatten → Dense(10)**; ghi kích thước qua từng lớp, số tham số và lý do chọn.
3. Không dùng pretrained model; xác định đầu vào/chuẩn hóa nhất quán, chỉ dùng augmentation nếu được nêu rõ và có so sánh công bằng.

**Khác Lab4:** phần CNN **không có sẵn** trong `Lab4.ipynb`; đây là nội dung bổ sung theo yêu cầu đồ án.


### 3.2. Cài đặt và kiểm tra các thành phần CNN

**TODO:**
1. Cài convolution/pooling, activation, classifier và **luồng gradient cập nhật trọng số**; kiểm tra shape và forward trên mini-batch.
2. **Tuân thủ PDF:** phụ lục yêu cầu tự cài tính toán lõi; PDF chỉ nói rõ được dùng PyTorch/TensorFlow cho **khung huấn luyện MLP**, không nói rõ cho CNN. Vì vậy **ưu tiên tự cài lõi CNN** hoặc hỏi giảng viên xác nhận phạm vi dùng framework **trước khi** sử dụng autograd/optimizer dựng sẵn cho CNN.
3. Test loss/gradient và dự đoán lớp 0–9 trên một batch nhỏ; không dùng test làm dữ liệu phát triển thuật toán.

**Kế thừa ý tưởng:** quy trình forward → loss → backward → update và kiểm tra gradient từ `Lab4`, nhưng cần cài riêng cho các phép tích chập/pooling.


### 3.3. Huấn luyện CNN và chọn checkpoint

**TODO:**
1. Viết vòng mini-batch training cho CNN và hàm validation theo epoch; đo thời gian fit như với MLP.
2. Vẽ train/validation loss và accuracy; theo dõi overfit, chọn epoch/checkpoint tốt nhất **bằng validation**.
3. Chọn cấu hình CNN đơn giản làm đối chứng; nếu thử nhiều CNN, báo rõ cấu hình và giữ phép so sánh công bằng.

**Gợi ý tái sử dụng:** phần logging, plotting, lưu best checkpoint và đo thời gian từ MLP/Lab4; **không sao chép phần backward Dense để thay backward Conv**.


### 3.4. Đánh giá CNN trên train / validation / test

**TODO:**
1. Dự đoán bằng **checkpoint CNN tốt nhất theo validation**, chỉ kiểm test khi đã khóa kiến trúc/siêu tham số.
2. Với **train, validation, test**, tính **accuracy, precision_macro, recall_macro, f1_macro** và vẽ **confusion matrix 10×10** cho từng tập.
3. Đo `train_seconds`, `predict_seconds` và ghi CPU/GPU để diễn giải tốc độ; xem ảnh ví dụ đúng/sai và lỗi phổ biến.

**Đầu ra:** metrics, confusion matrices và checkpoint CNN có thể dùng để so sánh với MLP.


## 4. So sánh và bàn giao Phase 2

### 4.1. So sánh MLP và CNN

**TODO:**
1. Tạo bảng **MLP vs CNN** trên **cùng tập train/validation/test**: accuracy, precision/recall/F1 macro, train time, predict time, số tham số.
2. Nhận xét **độ chính xác**, dấu hiệu overfit, chi phí tính toán và **khả năng giải thích**; phân tích vì sao CNN có thể tận dụng cấu trúc không gian 28×28 mà MLP flatten mất.
3. Đối chiếu 2–3 cặp chữ số bị nhầm và hình minh họa; kết luận có căn cứ từ **số liệu đo**, không khẳng định CNN luôn tốt hơn.

**Lưu ý:** bảng 4 mô hình **bắt buộc của PDF** (Naïve Bayes, Linear, MLP, Ensemble) sẽ tổng hợp ở Phase 3; CNN được thể hiện như mô hình **thứ 5 / nâng cao**.


### 4.2. Xuất kết quả chuẩn để Phase 3 đọc được

**TODO:**
1. Xuất `results/phase2_metrics.csv` với **6 dòng tối thiểu** (`MLP`, `CNN` × `train`, `validation`, `test`).
2. Dùng cùng schema Phase 1/3: `phase, model, split, accuracy, precision_macro, recall_macro, f1_macro, train_seconds, predict_seconds, n_samples, seed, params`.
3. Lưu checkpoint vào `models/`, biểu đồ/confusion matrix vào `figures/`; ghi tham chiếu `artifacts/split_indices.npz` + hash nguồn và **pipeline preprocessing riêng MLP/CNN** để tái lập.
4. Chạy kiểm tra notebook từ đầu; đảm bảo **độc lập với trạng thái kernel Phase 1**, không thiếu biến/đường dẫn.

**Kế thừa:** quy ước ở `plan.md`, `phase1.ipynb` và `TL_Lab03.ipynb` (bảng kết quả).


### 4.3. Nhận xét, nguồn tham khảo và checklist hoàn thành

**TODO:**
1. Viết tóm tắt công thức/pseudo-code **forward, backprop, SGD momentum, CNN**, và nhận xét từ biểu đồ/CM đủ dùng cho báo cáo PDF.
2. Ghi rõ phần kế thừa **`Lab4.ipynb`**, phần nhóm tự hoàn thiện, nguồn tham khảo và việc sử dụng AI (nếu có) theo phụ lục PDF.
3. Kiểm lại checklist bên dưới, không mở rộng sang các thí nghiệm **ngoài phạm vi bắt buộc** nếu chưa xong phần lõi.

**Checklist:**
- [ ] Đọc `artifacts/split_indices.npz` của Phase 1, đối chiếu hash/thứ tự ảnh và nhãn; không rò rỉ dữ liệu; có ảnh mẫu và kiểm shape.
- [ ] MLP tự cài backward + SGD momentum; có kiểm chứng gradient và đường học.
- [ ] ≥ 1 biểu đồ phân tích siêu tham số, chọn checkpoint bằng validation.
- [ ] MLP và CNN đều có **4 metric + confusion matrix trên cả 3 tập**.
- [ ] Có bảng **MLP–CNN** về độ chính xác, thời gian và khả năng giải thích.
- [ ] `results/phase2_metrics.csv` đúng schema, có checkpoints/plots và nguồn tham khảo.
